In [7]:
import os
import json
import psycopg2
import psycopg2.extras
from openai import OpenAI
from pgvector.psycopg2 import register_vector
from pathlib import Path
from dotenv import load_dotenv
root = Path.cwd()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Start the notebook from inside the cloned project.")
    root = root.parent

load_dotenv(root / "prod.env")

neon_connection_string = os.environ.get("NEON_DATABASE_URL")
openai_api_key = os.environ.get("OPENAI_KEY")

INPUT_DIR = root / "src" / "publications"
client = OpenAI(api_key=openai_api_key)
MODEL = "text-embedding-3-small"

In [8]:
def get_embedding(text):
    '''Get the embedding for a given text using the OpenAI API.'''
    response = client.embeddings.create(
        model=MODEL,
        input=text,
        dimensions=512
    )
    return response.data[0].embedding

In [14]:
def search_publications(prompt, top_k=5):
    '''Search for publications related to the given prompt using OpenAI embeddings.
    https://help.openai.com/en/articles/6824809-embeddings-faq'''
    embedding = get_embedding(prompt)
    conn = psycopg2.connect(neon_connection_string)
    register_vector(conn.cursor())
    cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cursor.execute("""
        SELECT  id, 
                source, 
                source_id, 
                publication_date, 
                doi, 
                url, 
                title, 
                abstract, 
                "references",
                authors,
                embedding <=> %s::vector AS distance
        FROM publications
        ORDER BY embedding <=> %s::vector
        LIMIT %s
    """, (embedding, embedding, top_k))
    results = cursor.fetchall()
    conn.close()
    return results

In [15]:
prompt = "Find Publications related to Colorectal Polyp Detection"
results = search_publications(prompt)

In [16]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame([dict(result) for result in results])

with pd.option_context("display.max_colwidth", 120):
    display(df)

,id,source,source_id,publication_date,doi,url,title,abstract,references,authors,distance
0,117029,semantic_scholar,25f696bdf01c125c416bbbaf6f85d8afd887bede,2020-08-01,10.1016/j.artmed.2020.101923,https://www.semanticscholar.org/paper/25f696bdf01c125c416bbbaf6f85d8afd887bede,Deep learning to find colorectal polyps in colonoscopy: A systematic literature review,"Colorectal cancer has a great incidence rate worldwide, but its early detection significantly increases the survival...",[],"{""L. F. Sánchez-Peralta"",""L. Bote-Curiel"",""A. Picón"",""F. Sánchez-Margallo"",""J. Pagador""}",0.332642
1,102237,semantic_scholar,357da6e2496a1f7e56421dc9154dc9c46f5d28f1,None,10.1016/j.neucom.2020.02.123,https://www.semanticscholar.org/paper/357da6e2496a1f7e56421dc9154dc9c46f5d28f1,Deep Neural Networks approaches for detecting and classifying colorectal polyps,Abstract Deep Learning (DL) has attracted a lot of attention in the field of medical image analysis because of its h...,[],"{""Alba Nogueira-Rodríguez"",""Rubén Domínguez-Carbajales"",""H. López-Fernández"",""Á. Iglesias"",""J. Cubiella"",""F. Fdez-Ri...",0.358674
2,90067,semantic_scholar,2e00ba630b8914e02d3d98ba6adf493bb98b244c,None,10.2139/ssrn.4111787,https://www.semanticscholar.org/paper/2e00ba630b8914e02d3d98ba6adf493bb98b244c,Improve Performance of Deep Learning Polyp Detection System in Clinical Colonoscopy,NaN,[],"{""W. Atuahene""}",0.365987
3,109843,openalex,W7163393344,2026-06-03,10.3389/frai.2026.1786679,https://doi.org/10.3389/frai.2026.1786679,"Deep learning driven colorectal polyp analysis: a review of detection, classification and segmentation methods",Colorectal polyps are key determinants of colorectal cancer. Their accurate detection during colonoscopy has been a ...,"[W2008359794, W2021088830, W2034269173, W2047717400, W2285968993, W2579357550, W2623808523, W2884985635, W2914568698...","{""Divya S"",""Sudha M""}",0.366347
4,22762,semantic_scholar,1f0003582dad3dd6ebbe216a85d72c80a20517be,2023-01-20,10.3390/s23031225,https://www.semanticscholar.org/paper/1f0003582dad3dd6ebbe216a85d72c80a20517be,Detection of Colorectal Polyps from Colonoscopy Using Machine Learning: A Survey on Modern Techniques,"Given the increased interest in utilizing artificial intelligence as an assistive tool in the medical sector, colore...",[],"{""Khaled ELKarazle"",""V. Raman"",""P. Then"",""Caslon Chua""}",0.372390


In [6]:
# Summarize Database content, quantity of total publications per source
def summarize_publications_by_source():
    conn = psycopg2.connect(neon_connection_string)
    cursor = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)
    cursor.execute("""
        SELECT source, COUNT(source_id) AS total_publications
        FROM publications
        GROUP BY source
        ORDER BY total_publications DESC
    """)
    results = cursor.fetchall()
    conn.close()
    return results

db_summary = summarize_publications_by_source()
for row in db_summary:
    print(f"Source: {row['source']}, Total Publications: {row['total_publications']}")

Source: openalex, Total Publications: 65262
Source: semantic_scholar, Total Publications: 47686
Source: arxiv, Total Publications: 4537
